Bakoulas Epameinondas - AEM: 10683

# Assignment 1

## Design of a weekly school schedule

First, we will import all the necessary data that we are given. We will also use the libraries `gurobipy` to solve the feasability problem and `pandas` to display our results.

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from IPython.display import display

departments = ['Dept1', 'Dept2']
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
time_slots = ['TS1_0800_1000', 'TS2_1015_1215', 'TS3_1400_1600', 'TS4_1615_1815']

# Structure: (Teacher, Subject, Hours_Dept1, Hours_Dept2)
assignments_data = [
    ('Gesmanidis', 'English', 1, 1),
    ('Insoulina', 'Biology', 3, 3),
    ('Chartoula', 'HistoryGeography', 2, 2),
    ('Lathopraxis', 'Math', 0, 4),
    ('Antiparagogos', 'Math', 4, 0),
    ('Kirkofidou', 'Physics', 3, 3),
    ('Platiazon', 'Philosophy', 1, 1),
    ('Bratsakis', 'PE', 1, 0),
    ('Trechalitoola', 'PE', 0, 1)
]

# Extract unique teachers and subjects
all_teachers = sorted(list(set(a[0] for a in assignments_data)))
all_subjects = sorted(list(set(a[1] for a in assignments_data)))

teacher_subject_combinations = []
for teacher_val, subject_val, *_ in assignments_data:
    teacher_subject_combinations.append((teacher_val, subject_val))


The decision variable X will be a **binary variable** that indicates if a teacher is going to teach a subject at a specific time slot on a specific day and on a specific department. Basically, this means that they will have the form

$$
X_{teacher,dept,subject,day,slot}
$$

We have 9 different teachers, 2 departments, 5 days, and 4 time slots in each day.

**Total number of combinations (variables X):** $$9 * 2 * 5 * 4 = 360$$

We're going to create the model first, then create the keys (variables) and insert them into the model.

In [2]:
model = gp.Model("SchoolTimetable")

# Key order: (teacher, department, subject, day, slot)
X_keys = []
for teacher, subject in teacher_subject_combinations:
    for day in days:
        for slot in time_slots:
            X_keys.append((teacher, 'Dept1', subject, day, slot))
            X_keys.append((teacher, 'Dept2', subject, day, slot))

print(f"Total X variables to be created: {len(X_keys)}")

# Create decision variables
X = model.addVars(X_keys, vtype=GRB.BINARY)

Set parameter Username
Set parameter LicenseID to value 2668025
Academic license - for non-commercial use only - expires 2026-05-19
Total X variables to be created: 360


### 1st constraint

The 1st constraint is that each teacher should teach for a specific number of hours (slots) in a week for each department.

This is expressed as:

$$\sum_{d \in \text{days}} \sum_{s \in \text{slots}} X_{\text{teacher}, \text{dept}, \text{subject}, d, s} = N$$

where $N$ is the number of hours that teacher should teach that subject in that department. 

This means that for each combination of (teacher, subject, dept), we need to loop through all the slots and days, sum the X variables and add that constraint.

In [3]:
for teacher, subject, hours_dept1, hours_dept2 in assignments_data:
        model.addConstr(
            gp.quicksum(X[teacher, 'Dept1', subject, day, slot]
                        for day in days
                        for slot in time_slots) == hours_dept1
        )
        model.addConstr(
            gp.quicksum(X[teacher, 'Dept2', subject, day, slot]
                        for day in days
                        for slot in time_slots) == hours_dept2
        )

### 2nd constraint

The 2nd constraint is that each teacher should not teach on the 2 departments at the same time.

This is expressed as:

$$\sum_{d \in \text{departments}} X_{\text{teacher}, d, \text{subject}, \text{day}, \text{slot}} \leq 1$$

This means that for each combination of (teacher, subject, day, slot), we need to loop through all the departments, sum them, and add that constraint.

In [4]:
for teacher, subj in teacher_subject_combinations:
    for day in days:
        for slot in time_slots:
            model.addConstr(
                gp.quicksum(X[teacher, dept, subj, day, slot]
                            for dept in departments) <= 1
            )

### 3rd constraint

The 3rd constraint is that each department should not have 2 different classes at the same time.

This is expressed as:

$$\sum_{(t,s) \in \text{teacher-subject-combinations}} X_{t, \text{dept}, s, \text{day}, \text{slot}} \leq 1$$

This means that for each combination of (dept, day, slot), we need to loop through all the entries (teacher, subject), sum them, and add that constraint.

In [5]:
for dept in departments:
    for day in days:
        for slot in time_slots:
            model.addConstr(
                gp.quicksum(X[teacher, dept, subject, day, slot]
                            for teacher, subject in teacher_subject_combinations) <= 1
            )


### 4th constraint

The 4th constraint is that the PE lessons should be done on Thursday at 14:00-16:00. 

We can just set the 2 specific variables refering to this specific constraint to 1.

In [6]:
pe_d1_key = ('Bratsakis', 'Dept1', 'PE', 'Thu', 'TS3_1400_1600')
model.addConstr(X[pe_d1_key] == 1)

pe_d2_key = ('Trechalitoola', 'Dept2', 'PE', 'Thu', 'TS3_1400_1600')
model.addConstr(X[pe_d2_key] == 1)

<gurobi.Constr *Awaiting Model Update*>

### 5th constraint

The 5th constraint is that no classes should be scheduled on Monday at 8:00-10:00.

So for the specific day (monday) and slot (8:00-10:00), we need to set all variables to 0.

In [7]:
day_val_constr5 = 'Mon'
slot_val_constr5 = 'TS1_0800_1000'

model.addConstr(
    gp.quicksum(X[teacher, dept, subject, day_val_constr5, slot_val_constr5]
                for teacher, subject in teacher_subject_combinations
                for dept in departments) == 0
)

<gurobi.Constr *Awaiting Model Update*>

### 6th constraint

The 6th constraint is that Mr. Lathopraxis should not teach on Monday morning (8:00-12:15).

We will manually add this constraint.

In [8]:
lathopraxis_abs_keys = [
    ('Lathopraxis', 'Dept2', 'Math', 'Mon', 'TS1_0800_1000'),
    ('Lathopraxis', 'Dept2', 'Math', 'Mon', 'TS2_1015_1215')
]
for key in lathopraxis_abs_keys:
    model.addConstr(X[key] == 0)

### 7th constraint

The 7th constraint is that Ms. Insoulina does not work on Wednesdays.

We will loop over all of the variables that have `teacher_val = 'Insoulina'`, `subject = 'Biology'` and `day = 'Wed'`.

In [9]:
teacher_val_constr7 = 'Insoulina'
subj_val_constr7 = 'Biology'
day_val_constr7 = 'Wed'

model.addConstr(
    gp.quicksum(X[teacher_val_constr7, dept, subj_val_constr7, day_val_constr7, slot] 
                for dept in departments
                for slot in time_slots) == 0
)

<gurobi.Constr *Awaiting Model Update*>

### 8th constraint

The 8th and last constraint is that no specific class (subject) should be taught twice in a single day.

This is expressed as:

$$\sum_{s \in \text{slots}} X_{\text{teacher}, \text{dept}, \text{subject}, \text{day}, s} \leq 1$$

This means that for each combination of (teacher, subject, dept, day), we need to loop through all the time slots, sum the X variables and ensure the sum is at most 1.

In [10]:
for teacher, subject in teacher_subject_combinations:
    for day in days:
        model.addConstr(
            gp.quicksum(X[teacher, 'Dept1', subject, day, slot_val]
                        for slot_val in time_slots) <= 1
        )
        model.addConstr(
            gp.quicksum(X[teacher, 'Dept2', subject, day, slot_val]
                        for slot_val in time_slots) <= 1
        )


Our goal is to just **find a feasible solution**, so we can just set the objective function to 0.

Basically this translates to "Minimize a constant function", i.e. don't optimize anything, just find a feasible solution.

After that, we optimize the model.

In [ ]:
model.setObjective(0, GRB.MINIMIZE)

model.optimize()

## Results

Finally, we display the results.

We start with displaying the **schedules for each department**.

In [12]:
active_assignments = []

if model.status == GRB.OPTIMAL or model.status == GRB.FEASIBLE:
    print("\nFeasible Timetable Found:\n")
    
    # Find which X values are equal to 1
    if model.SolCount > 0:
        for var_key in X.keys():
            if X[var_key].X == 1:
                teacher, dept, subject, day, slot = var_key
                active_assignments.append((teacher, dept, subject, day, slot))
    
    print("--- Department Schedules ---")
    for dept_to_display in departments:
        print(f"\n{dept_to_display} Schedule:")
        
        dept_df = pd.DataFrame(index=time_slots, columns=days)
        dept_df.fillna("FREE", inplace=True)
        
        for teacher, dept, subject, day, slot in active_assignments:
            if dept == dept_to_display:
                dept_df.at[slot, day] = f"{subject} ({teacher})"
        
        display(dept_df)
        
elif model.status == GRB.INFEASIBLE:
    print("\nThe model is infeasible. No solution exists under the given constraints.")



Feasible Timetable Found:

--- Department Schedules ---

Dept1 Schedule:


,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,HistoryGeography (Chartoula),Physics (Kirkofidou),Biology (Insoulina),Math (Antiparagogos)
TS2_1015_1215,Physics (Kirkofidou),Physics (Kirkofidou),Philosophy (Platiazon),HistoryGeography (Chartoula),FREE
TS3_1400_1600,Math (Antiparagogos),FREE,Math (Antiparagogos),PE (Bratsakis),English (Gesmanidis)
TS4_1615_1815,Biology (Insoulina),FREE,FREE,Math (Antiparagogos),Biology (Insoulina)



Dept2 Schedule:


,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,Biology (Insoulina),Math (Lathopraxis),FREE,Biology (Insoulina)
TS2_1015_1215,HistoryGeography (Chartoula),Math (Lathopraxis),FREE,Math (Lathopraxis),FREE
TS3_1400_1600,Biology (Insoulina),Philosophy (Platiazon),FREE,PE (Trechalitoola),Physics (Kirkofidou)
TS4_1615_1815,Math (Lathopraxis),Physics (Kirkofidou),Physics (Kirkofidou),English (Gesmanidis),HistoryGeography (Chartoula)


We will also display the **schedules for each teacher**.

In [13]:
if model.status == GRB.OPTIMAL or model.status == GRB.FEASIBLE:    
    print("\n--- Teacher Schedules ---")
    for teacher_to_display in all_teachers:
        print(f"\n{teacher_to_display} Schedule:")
        
        teacher_df = pd.DataFrame(index=time_slots, columns=days)
        teacher_df.fillna("FREE", inplace=True)
        
        for teacher, dept, subject, day, slot in active_assignments:
            if teacher == teacher_to_display:
                teacher_df.at[slot, day] = f"{subject} ({dept})"
        
        display(teacher_df)

elif model.status == GRB.INFEASIBLE:
    print("\nThe model is infeasible. No solution exists under the given constraints.")



--- Teacher Schedules ---

Antiparagogos Schedule:


,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,FREE,FREE,FREE,Math (Dept1)
TS2_1015_1215,FREE,FREE,FREE,FREE,FREE
TS3_1400_1600,Math (Dept1),FREE,Math (Dept1),FREE,FREE
TS4_1615_1815,FREE,FREE,FREE,Math (Dept1),FREE



Bratsakis Schedule:


,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,FREE,FREE,FREE,FREE
TS2_1015_1215,FREE,FREE,FREE,FREE,FREE
TS3_1400_1600,FREE,FREE,FREE,PE (Dept1),FREE
TS4_1615_1815,FREE,FREE,FREE,FREE,FREE



Chartoula Schedule:


...

,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,FREE,Math (Dept2),FREE,FREE
TS2_1015_1215,FREE,Math (Dept2),FREE,Math (Dept2),FREE
TS3_1400_1600,FREE,FREE,FREE,FREE,FREE
TS4_1615_1815,Math (Dept2),FREE,FREE,FREE,FREE



Platiazon Schedule:


,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,FREE,FREE,FREE,FREE
TS2_1015_1215,FREE,FREE,Philosophy (Dept1),FREE,FREE
TS3_1400_1600,FREE,Philosophy (Dept2),FREE,FREE,FREE
TS4_1615_1815,FREE,FREE,FREE,FREE,FREE



Trechalitoola Schedule:


,Mon,Tue,Wed,Thu,Fri
TS1_0800_1000,FREE,FREE,FREE,FREE,FREE
TS2_1015_1215,FREE,FREE,FREE,FREE,FREE
TS3_1400_1600,FREE,FREE,FREE,PE (Dept2),FREE
TS4_1615_1815,FREE,FREE,FREE,FREE,FREE
